In [2]:
# Anna code modified to run (imported modules and functions)

# import modules 
#===================================================================================================
import numpy as np, matplotlib.pyplot as plt, pandas as pd, pynbody, os
import scipy, timeit
plt.rcParams.update(plt.rcParamsDefault)

# This must be done BEFORE importing tangos
# Set the path to your TANGOS database
path = '/home/amolle/Marvel_recent.db'
os.environ['TANGOS_DB_CONNECTION'] = path

import tangos 

# pick sim
sims = tangos.all_simulations()
print("simulations found:", sims)

# given rho(r) arrays and target, returns closest point + slope 
def find_point(r, den, target, array_name):

    r = np.array(r)
    den = np.array(den)

    # find slope array
    slope = np.gradient(np.log10(den), np.log10(r))

    # define array_name as correlated array
    arrays = {'r': r, 'den': den, 'slope': slope}

    if array_name not in arrays:
        print('array_name not recognized')

    # find index of closest point
    idx = np.argmin(np.abs(arrays[array_name] - target))

    # return values at that index 
    return r[idx], den[idx], slope[idx]

def fit_cEinasto_prof(r, rho2, r2, rc):
    alpha_ep = 0.16
    return rho2 * np.exp((-2/alpha_ep) * ( ((r + rc)/r2)**(alpha_ep) - 1 ))
    
def cEinasto_logslope(r, rs, rc):
    alpha_ep = 0.16
    return -2 * (r/rs)**alpha_ep * (1 + (rc/r))**(alpha_ep-1)

def cEinasto_fit(halo, b, r, den):
    if(len(b) > 1):
        fit, cov = scipy.optimize.curve_fit(fit_cEinasto_prof, r, den, bounds=b)
    else:
        fit, cov = scipy.optimize.curve_fit(fit_cEinasto_prof, r, den)
    gtz_denmap = den > 0
    r = r[gtz_denmap]
    den = den[gtz_denmap]
    return fit, cov, r, den
    
def fit_NFW_prof(r, rho_0, r_s):
    return rho_0/((r/r_s)*(1 + (r/r_s))**2)
    
def NFW_fit(halo, b, r, den):
    if(len(b) > 0):
        fit, cov = scipy.optimize.curve_fit(fit_NFW_prof, r, den, bounds=b)
    else:
        fit, cov = scipy.optimize.curve_fit(fit_NFW_prof, r, den)
    gtz_denmap = den > 0
    r = r[gtz_denmap]
    den = den[gtz_denmap]
    return fit, cov, r, den
    
def fit_Einasto_prof(r, rho2, r2):
    alpha_ep = 0.16
    return rho2 * np.exp((-2/alpha_ep) * ((r/r2)**(alpha_ep) - 1 ))
    
def Einasto_fit(halo, b, r, den):
    if(len(b) > 0):
        fit, cov = scipy.optimize.curve_fit(fit_Einasto_prof, r, den, bounds=b)
    else:
        fit, cov = scipy.optimize.curve_fit(fit_Einasto_prof, r, den)
    gtz_denmap = den > 0
    r = r[gtz_denmap]
    den = den[gtz_denmap]
    return fit, cov, r, den
    
def curvefits(model, output, halo, baryon, fit_type, bounds, family, r, den):
    if fit_type == 'cEinasto':
        fit, cov, r, den = cEinasto_fit(halo, bounds, r, den)
        den_fit = fit_cEinasto_prof(r, fit[0], fit[1], fit[2])
        den_fit = np.array(den_fit)
        den = np.array(den)
        residue_den = (np.log10(den_fit)-np.log10(den))**2
        Q = (sum(residue_den) / len(den))**(1/2)
    elif fit_type == 'NFW':
        fit, cov, r, den = NFW_fit(halo, bounds, r, den)
        den_fit = fit_NFW_prof(r, fit[0], fit[1])
        den_fit = np.array(den_fit)
        den = np.array(den)
        residue_den = (np.log10(den_fit)-np.log10(den))**2
        Q = (sum(residue_den) / len(den))**(1/2)
    elif fit_type == 'Einasto':
        fit, cov, r, den = Einasto_fit(halo, bounds, r, den)
        den_fit = fit_Einasto_prof(r, fit[0], fit[1])
        den_fit = np.array(den_fit)
        den = np.array(den)
        residue_den = (np.log10(den_fit)-np.log10(den))**2
        Q = (sum(residue_den) / len(den))**(1/2)
    return fit, Q, residue_den
    
def optim_fit(model, fit_type, output, min_rs, baryon, family, ID, r, den, rvir, sim):

    r = np.array(r)
    den = np.array(den)
    
    outfn = f"cvir_outputs/fits_output_sim_{sim}_ID_{ID}.txt"
    outfile = open(outfn, 'w')
    outfile.write('Q min_rs rhos rs[kpc] rc[kpc] rvir[kpc]\n')

    # resolution limits 
    mask = (r >= 0.25) & (r <= rvir)
    r = r[mask]
    den = den[mask]

    den_l = 1e4
    den_h = 1e10
    r_h = rvir

    for mrs in min_rs:
        if fit_type == 'cEinasto':
            bounds    = ([den_l, mrs, 0],[den_h, r_h, 20])
            fitparams, Q, residue_den= curvefits(model, output, halo, baryon, fit_type, bounds, family, r, den)
            rhos = fitparams[0]
            rs = fitparams[1]
            rc = fitparams[2]
            slope = cEinasto_logslope(rs, rs, rc)
            print(f'bounds: [{mrs},{r_h}] rs: {rs:.3f} rc: {rc:.3f} slp: {slope:.3f} Q: {Q:.3f}')
            
        elif fit_type == 'NFW':
            bounds = ([den_l, mrs],[den_h, r_h])
            fitparams, Q, residue_den= curvefits(model, output, halo, baryon, fit_type, bounds, family, r, den)
            rhos = fitparams[0]
            rs = fitparams[1]
            rc = 0
            
        elif fit_type == 'Einasto':
            bounds = ([den_l, mrs],[den_h, r_h])
            fitparams, Q, residue_den= curvefits(model, output, halo, baryon, fit_type, bounds, family, r, den)
            rhos = fitparams[0]
            rs = fitparams[1]
            rc = 0
        outfile.write('%.3f %.3f %.3f %.3f %.3f %.3f\n' %(Q, mrs, rhos, rs, rc, rvir))
    outfile.close()

    #print(f'halo:{ID} rhos = {rhos:.3e} rs = {rs:.3f} mrs = {mrs:.3f} rvir = {rvir:.3f}')
    return Q, mrs, rhos, rs, rc, rvir


simulations found: [<Simulation("cptmarvel.cosmo25cmb.4096g5HbwK1BH")>, <Simulation("elektra.cosmo25cmb.4096g5HbwK1BH")>, <Simulation("rogue.cosmo25cmb.4096g5HbwK1BH")>, <Simulation("storm.cosmo25cmb.4096g5HbwK1BH")>]


In [2]:
# Anna code modified (run vbounds sims and create data) - basic
#======================================================================================================

# load sim
sim = tangos.all_simulations()[3]
sim_name = sim.basename

# constants
output = '004096'
fit_type = 'cEinasto'
baryon = False
family = 'dm'
model = 'SIDM'

halos = sim.timesteps[-1].halos

Halo_list = [str(h.id + 1) for h in halos]
print(f'end halo = {Halo_list[-1]}, Q goal: 0.07')

# start of main program 
start = timeit.default_timer()

Q_avg = []

for halo in halos:

    HID = str(halo.id)

    if int(HID) % 25 == 0:
        print(f'  current ID: {HID}')
    
    if 'dm_density_profile' in halo.keys():
        
        den = halo['dm_density_profile']
        rvir = halo["max_radius"]
        vbounds = np.linspace(0.25, rvir/5, 15)
     
        # find step size
        dr = rvir / len(den)

        # create r array (make bin edges then subtract dr/2 to get centers)
        r = np.linspace(dr , len(den)*dr, len(den)) - (dr/2)

        try:
            optim_fit(model, fit_type, output, vbounds, baryon, family, HID, r[:-10], den[:-10], rvir, sim_name)
            print(f'ID:{HID} Mvir = {halo['Mvir']:.3e}')
            
            fits = f"cvir_outputs/fits_output_sim_{sim_name}_ID_{HID}.txt"

            fit_df = pd.read_table(fits, sep=r'\s+')

            # find best fit 
            Q_vals = fit_df['Q'].to_numpy()
            min_ind = np.argmin(Q_vals)

            # get best fit params 
            rs   = fit_df['rs[kpc]'].to_numpy()
            rc   = fit_df['rc[kpc]'].to_numpy()
            rs_i = float(rs[min_ind])
            rc_i = float(rc[min_ind])

            Q_avg_i = np.min(Q_vals)
            Q_avg.append(Q_avg_i)
            #print(f"ID {HID}: Q = {Q_avg_i:.4f}")
        
        except RuntimeError:
            print('error')

    else:
        Mvir = halo['Mvir']
        #print(f'halo:{HID} no dm profile. Mvir = {Mvir}')
        continue 

Q_sim = np.median(Q_avg)
stop = timeit.default_timer()
run_time = stop - start

print(f' -----------------------> Q_sim = {Q_sim}')
print('run time optim_fit =   ',run_time)


end halo = 246833, Q goal: 0.07
bounds: [0.25,86.03479901858363] rs: 0.539 rc: 6.190 slp: -0.240 Q: 0.335
bounds: [1.4612114145511945,86.03479901858363] rs: 1.461 rc: 4.701 slp: -0.597 Q: 0.111
bounds: [2.672422829102389,86.03479901858363] rs: 2.672 rc: 3.929 slp: -0.936 Q: 0.081
bounds: [3.8836342436535833,86.03479901858363] rs: 3.884 rc: 3.498 slp: -1.166 Q: 0.146
bounds: [5.094845658204778,86.03479901858363] rs: 5.095 rc: 3.205 slp: -1.327 Q: 0.201
bounds: [6.306057072755973,86.03479901858363] rs: 6.306 rc: 2.987 slp: -1.444 Q: 0.244
bounds: [7.517268487307167,86.03479901858363] rs: 7.517 rc: 2.814 slp: -1.531 Q: 0.280
bounds: [8.728479901858362,86.03479901858363] rs: 8.728 rc: 2.673 slp: -1.598 Q: 0.311
bounds: [9.939691316409556,86.03479901858363] rs: 9.940 rc: 2.554 slp: -1.650 Q: 0.337
bounds: [11.15090273096075,86.03479901858363] rs: 11.151 rc: 2.451 slp: -1.693 Q: 0.360
bounds: [12.362114145511946,86.03479901858363] rs: 12.362 rc: 2.361 slp: -1.727 Q: 0.381
bounds: [13.5733255

KeyboardInterrupt: 

In [6]:
# Anna code modified (read data table + find cvir + write in sim specific output file) only Qmin

# constants
output = '004096'
fit_type = 'cEinasto'
baryon = False
family = 'dm'
model = 'SIDM'
db = 'Marvel'

n_sim = 4
error_file = f'cvir_results/cvir_{fit_type}_{db}.txt'
ferr = open(error_file, 'w')
ferr.write(f'Error log DB:{db}\n')

for sim_id in range(n_sim):
    Nan_count = 0
    sim = tangos.all_simulations()[sim_id]
    sim_name = sim.basename
    
    halos = sim.timesteps[-1].halos
    halo_ids = [str(h.id + 1) for h in halos]
    halo_ids = halo_ids[0] - 
    Mvir_array = np.array([h['Mvir'] for h in halos])
    
    print(f"\nProcessing sim {sim_id}: {sim}")

    # Open the output file for writing
    out_file = f'cvir_results/cvir_{fit_type}_{sim_name}.txt'
    f = open(out_file, 'w')
    f.write('halos\tcvir\n')

    for ID_index, ID in enumerate(halo_ids):
            
        filepath = f"cvir_outputs/fits_output_sim_{sim_name}_ID_{ID}.txt"
        if not os.path.exists(filepath):
            print(f"------> File not found for ID {ID}, skipping.", file=ferr)
            cvir = "NaN"
            f.write(f'{ID}\t{cvir}\n')
            Nan_count += 1
            continue
            
        try:
            fit_df = pd.read_table(filepath, sep=r'\s+')
    
            Q    = fit_df['Q'].to_numpy()
            rs   = fit_df['rs[kpc]'].to_numpy()
            rc   = fit_df['rc[kpc]'].to_numpy()
            rvir = fit_df['rvir[kpc]'].to_numpy()
            rhos = fit_df['rhos'].to_numpy()

            min_ind = np.argmin(Q)
            Q_min = float(Q[min_ind])

            if Q_min > 0.4:
                print(f'Error ID:{ID}\tMvir = {Mvir_array[ID_index]:.3e}  High Q value Q={Q_min}', file=ferr)
                cvir = "NaN"
                f.write(f'{ID}\t{cvir}\n')
                Nan_count += 1
                continue 

            rs_i = float(rs[min_ind])
            rc_i = float(rc[min_ind])

            if rc_i > rs_i:
                print(f'Error ID:{ID}\tMvir = {Mvir_array[ID_index]:.3e}  rc > rs', file=ferr)
                cvir = "NaN"
                f.write(f'{ID}\t{cvir}\n')
                Nan_count += 1
                continue 

            rho_i = float(rhos[min_ind])
            r200_i = rvir[min_ind]
    
            r = np.linspace(0.25, r200_i, 100)
            m2_slope = cEinasto_logslope(r, rs_i, rc_i) + 2
            m2_slope_ind = np.argmin(np.abs(m2_slope))
            r2 = r[m2_slope_ind]
    
            cvir = r200_i / r2
            f.write(f'{ID}\t{cvir}\n')

        except Exception as e:
            print(f" Error ID:{ID} no output table found.", file=ferr)
            cvir = "NaN"
            f.write(f'{ID}\t{cvir}\n')
            Nan_count += 1

    print(f'sim {sim_name} contains {Nan_count} NaN values / {len(halo_ids)}\n\n', file=ferr)

f.close()
ferr.close()


halo ids: [   0   -1   -2   -3   -4   -5   -6   -7   -8   -9  -10  -11  -12  -13
  -14  -15  -16  -17  -18  -19  -20  -21  -22  -23  -24  -25  -26  -27
  -28  -29  -30  -31  -32  -33  -34  -35  -36  -37  -38  -39  -40  -41
  -42  -43  -44  -45  -46  -47  -48  -49  -50  -51  -52  -53  -54  -55
  -56  -57  -58  -59  -60  -61  -62  -63  -64  -65  -66  -67  -68  -69
  -70  -71  -72  -73  -74  -75  -76  -77  -78  -79  -80  -81  -82  -83
  -84  -85  -86  -87  -88  -89  -90  -91  -92  -93  -94  -95  -96  -97
  -98  -99 -100 -101 -102 -103 -104 -105 -106 -107 -108 -109 -110 -111
 -112 -113 -114 -115 -116 -117 -118 -119 -120 -121 -122 -123 -124 -125
 -126 -127 -128 -129 -130 -131 -132 -133 -134 -135 -136 -137 -138 -139
 -140 -141 -142 -143 -144 -145 -146 -147 -148 -149 -150 -151 -152 -153
 -154 -155 -156 -157 -158 -159 -160 -161 -162 -163 -164 -165 -166 -167
 -168 -169 -170 -171 -172 -173 -174 -175 -176 -177 -178 -179 -180 -181
 -182 -183 -184 -185 -186 -187 -188 -189 -190 -191 -192 -193 -194 -

In [3]:
# Anna code modified (read data table + find cir + write in sim specific output file) pick fit

# constants
output = '004096'
fit_type = 'cEinasto'
baryon = False
family = 'dm'
model = 'SIDM'
db = 'DCJL'

n_sim = 4
error_file = f'cvir_results/cvir_{fit_type}_{db}.txt'
ferr = open(error_file, 'w')
ferr.write(f'Error log DB:{db}\n')

for sim_id in range(n_sim):
    Nan_count = 0
    sim = tangos.all_simulations()[sim_id]
    sim_name = sim.basename
    
    halos = sim.timesteps[-1].halos
    halo_ids = [str(h.id + 1) for h in halos]
    Mvir_array = np.array([h['Mvir'] for h in halos])
    
    print(f"\nProcessing sim {sim_id}: {sim}")

    # Open the output file for writing
    out_file = f'cvir_results/cvir_{fit_type}_{sim_name}.txt'
    f = open(out_file, 'w')
    f.write('halos\tcvir\n')

    for ID_index, ID in enumerate(halo_ids):
            
        filepath = f"cvir_outputs/fits_output_sim_{sim_name}_ID_{ID}.txt"
        if not os.path.exists(filepath):
            print(f"------> File not found for ID {ID}, skipping.", file=ferr)
            cvir = "NaN"
            f.write(f'{ID}\t{cvir}\n')
            Nan_count += 1
            continue
            
        try:
            fit_df = pd.read_table(filepath, sep=r'\s+')
    
            Q    = fit_df['Q'].to_numpy()
            rs   = fit_df['rs[kpc]'].to_numpy()
            rc   = fit_df['rc[kpc]'].to_numpy()
            rvir = fit_df['rvir[kpc]'].to_numpy()
            rhos = fit_df['rhos'].to_numpy()

            # Build a mask for "good" fits
            good_mask = (rc <= rs) & (Q <= 0.4)
            
            # Apply mask
            good_indices = np.where(good_mask)[0]
            
            if len(good_indices) == 0:
                # No fit passes both checks
                print(f'Error ID:{ID}\tMvir = {Mvir_array[ID_index]:.3e}  No valid fit found (rc <= rs and Q <= 0.4)', file=ferr)
                f.write(f'{ID}\tNaN\n')
                Nan_count += 1
                continue
            
            # From the good fits, choose the one with the lowest Q
            chosen_idx = good_indices[np.argmin(Q[good_indices])]
            
            # Extract parameters for chosen fit
            rs_i  = float(rs[chosen_idx])
            rc_i  = float(rc[chosen_idx])
            rho_i = float(rhos[chosen_idx])
            r200_i = float(rvir[chosen_idx])
            
            # Compute cvir
            r = np.linspace(0.25, r200_i, 100)
            m2_slope = cEinasto_logslope(r, rs_i, rc_i) + 2
            r2 = r[np.argmin(np.abs(m2_slope))]
    
            cvir = r200_i / r2
            f.write(f'{ID}\t{cvir}\n')

        except Exception as e:
            print(f" Error ID:{ID} no output table found.", file=ferr)
            cvir = "NaN"
            f.write(f'{ID}\t{cvir}\n')
            Nan_count += 1

    print(f'sim {sim_name} contains {Nan_count} NaN values / {len(halo_ids)}\n\n', file=ferr)

f.close()
ferr.close()



Processing sim 0: <Simulation("h148.cosmo50PLK.3072g3HbwK1BH")>

Processing sim 1: <Simulation("h229.cosmo50PLK.3072gst5HbwK1BH")>

Processing sim 2: <Simulation("h242.cosmo50PLK.3072gst5HbwK1BH")>

Processing sim 3: <Simulation("h329.cosmo50PLK.3072gst5HbwK1BH")>


In [80]:
# Unchanged code from Anna 
output = '004096'
fit_type = 'NFW'
vbounds = np.linspace(0.01, 10, 100)
model = 'SIDM'
Halo_list = np.loadtxt("IDs", delimiter=',')
for HID in Halo_list[2:3]:
    new_HID = int(HID) - 1
    print(HID, new_HID)
    halo = tangos.get_simulation("storm.cosmo25cmbsi2s50v35.4096").timesteps[-1].halos[new_HID]
    density = halo.calculate_for_progenitors('dm_density_profile')
    rvirial = halo.calculate_for_progenitors("max_radius")[0]
    index = 0
    for den in density[0]:
        hID = str(HID)
        rvir = rvirial[index]
        dr = 0.06498287916656409
        r = np.linspace(dr , len(den)*dr, len(den)) - (dr/2)
        try:
            optim_fit(model, fit_type, output, vbounds, baryon, family, hID, r[:-10], den[:-10], rvir, index)
            index+=1
            #print(index)
        except RuntimeError:
            index+=1

import pynbody
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
plt.rcParams.update(plt.rcParamsDefault)

def fit_cEinasto_prof(r, rho2, r2, rc):
    alpha_ep = 0.16
    return rho2 * np.exp((-2/alpha_ep) * ( ((r + rc)/r2)**(alpha_ep) - 1 ))
    
def cEinasto_logslope(r, rho, rs, rc):
    alpha_ep = 0.16
    return -2 * (r/rs)**alpha_ep * (1 + (rc/r))**(alpha_ep-1)

def cEinasto_fit(halo, b, r, den):
    if(len(b) > 1):
        fit, cov = scipy.optimize.curve_fit(fit_cEinasto_prof, r, den, bounds=b)
    else:
        fit, cov = scipy.optimize.curve_fit(fit_cEinasto_prof, r, den)
    gtz_denmap = den > 0
    r = r[gtz_denmap]
    den = den[gtz_denmap]
    return fit, cov, r, den
    
def fit_NFW_prof(r, rho_0, r_s):
    return rho_0/((r/r_s)*(1 + (r/r_s))**2)
    
def NFW_fit(halo, b, r, den):
    if(len(b) > 0):
        fit, cov = scipy.optimize.curve_fit(fit_NFW_prof, r, den, bounds=b)
    else:
        fit, cov = scipy.optimize.curve_fit(fit_NFW_prof, r, den)
    gtz_denmap = den > 0
    r = r[gtz_denmap]
    den = den[gtz_denmap]
    return fit, cov, r, den
    
def fit_Einasto_prof(r, rho2, r2):
    alpha_ep = 0.16
    return rho2 * np.exp((-2/alpha_ep) * ((r/r2)**(alpha_ep) - 1 ))
    
def Einasto_fit(halo, b, r, den):
    if(len(b) > 0):
        fit, cov = scipy.optimize.curve_fit(fit_Einasto_prof, r, den, bounds=b)
    else:
        fit, cov = scipy.optimize.curve_fit(fit_Einasto_prof, r, den)
    gtz_denmap = den > 0
    r = r[gtz_denmap]
    den = den[gtz_denmap]
    return fit, cov, r, den
    
def curvefits(model, output, halo, baryon, fit_type, bounds, family, r, den):
    if fit_type == 'cEinasto':
        fit, cov, r, den = cEinasto_fit(halo, bounds, r, den)
        den_fit = fit_cEinasto_prof(r, fit[0], fit[1], fit[2])
        den_fit = np.array(den_fit)
        den = np.array(den)
        residue_den = (np.log10(den_fit)-np.log10(den))**2
        Q = (sum(residue_den) / len(den))**(1/2)
    elif fit_type == 'NFW':
        fit, cov, r, den = NFW_fit(halo, bounds, r, den)
        den_fit = fit_NFW_prof(r, fit[0], fit[1])
        den_fit = np.array(den_fit)
        den = np.array(den)
        residue_den = (np.log10(den_fit)-np.log10(den))**2
        Q = (sum(residue_den) / len(den))**(1/2)
    elif fit_type == 'Einasto':
        fit, cov, r, den = Einasto_fit(halo, bounds, r, den)
        den_fit = fit_Einasto_prof(r, fit[0], fit[1])
        den_fit = np.array(den_fit)
        den = np.array(den)
        residue_den = (np.log10(den_fit)-np.log10(den))**2
        Q = (sum(residue_den) / len(den))**(1/2)
    return fit, Q, residue_den
    
def optim_fit(model, fit_type, output, min_rs, baryon, family, ID, r, den, rvir, h_index):
    outfn = 'DM_trace/' + model + '_fit_type_' + fit_type + '_output_' + output +  '_halo_'+str(ID)+'_' + str(h_index)+ '_fits.dat'
    outfile = open(outfn, 'w')
    outfile.write('Q min_rs rhos rs[kpc] rc[kpc] rvir[kpc]\n')
    rmax = 'rvir'
    gtz_denmap = r < rvir
    r = r[gtz_denmap]
    den = den[gtz_denmap]
    gtz_denmap = r > 0.39
    r = r[gtz_denmap]
    den = den[gtz_denmap]
    for mrs in min_rs:
        if fit_type == 'cEinasto':
            bounds    = ([1e4, mrs, 0],[1e10, 100, 20])
            #bounds = np.array([0])
            fitparams, Q, residue_den= curvefits(model, output, halo, baryon, fit_type, bounds, family, r, den)
            Q = (sum(residue_den) / len(residue_den))**(1/2)
            rhos = fitparams[0]
            rs = fitparams[1]
            rc = fitparams[2]
        elif fit_type == 'NFW':
            bounds = ([1e6, mrs],[1e10, 100])
            fitparams, Q, residue_den= curvefits(model, output, halo, baryon, fit_type, bounds, family, r, den)
            Q = (sum(residue_den) / len(residue_den))**(1/2)
            rhos = fitparams[0]
            rs = fitparams[1]
            rc = 0
        elif fit_type == 'Einasto':
            bounds = ([1e6, mrs],[1e10, 100])
            fitparams, Q, residue_den= curvefits(model, output, halo, baryon, fit_type, bounds, family, r, den)
            Q = (sum(residue_den) / len(residue_den))**(1/2)
            rhos = fitparams[0]
            rs = fitparams[1]
            rc = 0
        outfile.write('%.3f %.3f %.3f %.3f %.3f %.3f\n' %(Q, mrs, rhos, rs, rc, rvir))
    outfile.close()
    return Q, mrs, rhos, rs, rc, rvir
    

num = 0
#c_ass_L = []
rho_s = []
r_s = []
for ID in Halo_list:
    print(ID)
    new_HID = int(ID) - 1
    halo = tangos.get_simulation("storm.cosmo25cmbsi2s50v35.4096").timesteps[-1].halos[new_HID]
    output = '004096'
    fit_type = 'NFW'
    baryon = False
    family = 'dm'
    model = 'SIDM'
    dr = 0.06498287916656409
    #index = interger[num]
    rs_array = []
    rhos_array = []
    for index in range(36):
        fit_df = pd.read_table('DM_trace/' + model + '_fit_type_' + fit_type + '_output_' + output +  '_halo_' + str(ID) +'_' + str(index) + '_fits.dat', sep='\s+')
        Q    = fit_df['Q'].to_numpy()
        rs   = fit_df['rs[kpc]'].to_numpy()
        rc   = fit_df['rc[kpc]'].to_numpy()
        rvir = fit_df['rvir[kpc]'].to_numpy()
        rhos = fit_df['rhos'].to_numpy()
        min_ind = np.argmin(Q)
        Q_i  = Q[min_ind]
        rs_i = rs[min_ind]
        rc_i = rc[min_ind]
        rho_i = rhos[min_ind]
        r200_i = rvir[min_ind]
        rs_array.append(rs_i)
        rhos_array.append(rho_i)
        #print("cEinasto", rho_i)
        r = np.linspace(0.25, r200_i, 100)
        m2_slope = cEinasto_logslope(r, rho_i, rs_i, rc_i) + 2
        m2_slope_ind = np.argmin(np.abs(m2_slope))
        r2 = r[m2_slope_ind]
        #c200_i = r200_i / r2
        #concentration = R_ass[num]/r2
        #c_ass_L.append(concentration)
    rho_s.append(rhos_array)
    r_s.append(rs_array)
        #num+=1



<>:159: SyntaxWarning: invalid escape sequence '\s'
<>:159: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_701/3893594534.py:159: SyntaxWarning: invalid escape sequence '\s'
  fit_df = pd.read_table('DM_trace/' + model + '_fit_type_' + fit_type + '_output_' + output +  '_halo_' + str(ID) +'_' + str(index) + '_fits.dat', sep='\s+')
/tmp/ipykernel_701/3893594534.py:159: SyntaxWarning: invalid escape sequence '\s'
  fit_df = pd.read_table('DM_trace/' + model + '_fit_type_' + fit_type + '_output_' + output +  '_halo_' + str(ID) +'_' + str(index) + '_fits.dat', sep='\s+')


FileNotFoundError: IDs not found.